In [ ]:
# CELL 1 — Clean Kaggle environment and GPU verification

import os
import json
import math
import random
import shutil
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageEnhance
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

print("PyTorch location:", getattr(torch, "__file__", "unknown"))
print("PyTorch version :", getattr(torch, "__version__", "unknown"))

if not hasattr(torch, "device"):
    raise RuntimeError(
        "PyTorch is not loaded correctly. Restart the Kaggle session and do not install/upgrade torch."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. In Kaggle select a T4 GPU accelerator, restart the session, and run again."
    )

DEVICE = torch.device("cuda:0")

print("Selected device :", DEVICE)
print("GPU name        :", torch.cuda.get_device_name(0))
print("CUDA version    :", torch.version.cuda)
print("Compute capability:", torch.cuda.get_device_capability(0))

try:
    test_tensor = torch.tensor([1.0, 2.0, 3.0], device=DEVICE)
    test_result = (test_tensor * 2).cpu()
    print("CUDA test result:", test_result.tolist())
except Exception as error:
    raise RuntimeError(
        "CUDA execution failed. Restart the Kaggle session with a T4 GPU and do not reinstall torch/torchvision/Pillow."
    ) from error

print("\nCell 1 completed successfully.")


In [ ]:
# CELL 2 — Find TWO ZIP datasets in Kaggle, extract them, and configure training

KAGGLE_INPUT = Path("/kaggle/input")
EXTRACT_ROOT = Path("/kaggle/working/maize_two_datasets_extracted")
RESULTS_ROOT = Path("/kaggle/working/maize_tassel_counting_results")

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# OPTIONAL MANUAL MODE
# ------------------------------------------------------------------
# Normally leave this empty. The notebook will automatically find two ZIPs
# anywhere under /kaggle/input.
#
# If Kaggle contains more than two ZIP files, put the exact two paths here:
# MANUAL_ZIP_PATHS = [
#     "/kaggle/input/dataset-one/data1.zip",
#     "/kaggle/input/dataset-two/data2.zip",
# ]
MANUAL_ZIP_PATHS = []

if MANUAL_ZIP_PATHS:
    zip_paths = [Path(path) for path in MANUAL_ZIP_PATHS]
else:
    zip_paths = sorted(
        path for path in KAGGLE_INPUT.rglob("*.zip")
        if path.is_file()
    )

print("ZIP files found in Kaggle:")
for path in zip_paths:
    print(" -", path)

if len(zip_paths) != 2:
    raise RuntimeError(
        f"Expected exactly 2 ZIP datasets, but found {len(zip_paths)}. "
        "If Kaggle contains other ZIP files, set MANUAL_ZIP_PATHS in Cell 2 to the exact two dataset ZIPs."
    )

for path in zip_paths:
    if not path.exists():
        raise FileNotFoundError(f"ZIP not found: {path}")
    if not zipfile.is_zipfile(path):
        raise RuntimeError(f"Not a valid ZIP file: {path}")

# Remove an old extraction so stale files can never mix with the new run.
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)


def safe_extract_zip(zip_path, destination):
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(
                    f"Unsafe path inside ZIP {zip_path.name}: {member.filename}"
                )
        archive.extractall(destination)


DATASETS = []

for dataset_number, zip_path in enumerate(zip_paths, start=1):
    source_name = f"dataset_{dataset_number}_{zip_path.stem}"
    extract_dir = EXTRACT_ROOT / source_name

    print(f"\nExtracting {zip_path.name} -> {extract_dir}")
    safe_extract_zip(zip_path, extract_dir)

    image_count = sum(
        1 for p in extract_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
        }
    )
    mat_count = sum(1 for p in extract_dir.rglob("*.mat") if p.is_file())

    DATASETS.append({
        "source": source_name,
        "zip_path": zip_path,
        "root": extract_dir,
        "image_count": image_count,
        "mat_count": mat_count,
    })

print("\nDATASET SUMMARY")
print("=" * 72)
for dataset in DATASETS:
    print("Source      :", dataset["source"])
    print("ZIP         :", dataset["zip_path"])
    print("Extracted to:", dataset["root"])
    print("Images      :", dataset["image_count"])
    print("MAT files   :", dataset["mat_count"])
    print("-" * 72)

bad_sources = [
    d["source"] for d in DATASETS
    if d["image_count"] == 0 or d["mat_count"] == 0
]
if bad_sources:
    raise RuntimeError(
        "Each ZIP must contain images and MATLAB .mat point annotations for this density-counting pipeline. "
        f"Problem datasets: {bad_sources}"
    )

# Keep DATA_ROOTS for later summary/export cells.
DATA_ROOTS = [dataset["root"] for dataset in DATASETS]

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Training
CROP_SIZE = 512
OUTPUT_STRIDE = 8
CROPS_PER_IMAGE = 3
BATCH_SIZE = 4
NUM_WORKERS = 0
EPOCHS = 120
EARLY_STOPPING = 15

# Image resizing
TRAIN_MAX_SIDE = 1600
EVAL_MAX_SIDE = 1600

# Density-map settings
GAUSSIAN_SIGMA = 1.5
DENSITY_SCALE = 100.0

# Optimizer
FRONTEND_LR = 1e-5
BACKEND_LR = 1e-4
WEIGHT_DECAY = 1e-4

# Data augmentation
POINT_CENTERED_CROP_PROBABILITY = 0.70
HORIZONTAL_FLIP_PROBABILITY = 0.50

print("\nResults directory:", RESULTS_ROOT)
print("Cell 2 completed successfully.")


In [ ]:
# CELL 3 — Pair images with MAT annotations independently inside BOTH datasets

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}


def normalized_stem(path):
    stem = path.stem.lower()

    prefixes = [
        "gt_", "ground_truth_", "groundtruth_",
        "annotation_", "annotations_", "ann_"
    ]

    suffixes = [
        "_gt", "_ground_truth", "_groundtruth",
        "_annotation", "_annotations", "_ann"
    ]

    changed = True
    while changed:
        changed = False

        for prefix in prefixes:
            if stem.startswith(prefix):
                stem = stem[len(prefix):]
                changed = True

        for suffix in suffixes:
            if stem.endswith(suffix):
                stem = stem[:-len(suffix)]
                changed = True

    return stem


pairs = []
unmatched_images = []
source_pair_stats = []

for dataset in DATASETS:
    source = dataset["source"]
    root = dataset["root"]

    image_files = sorted(
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    mat_files = sorted(path for path in root.rglob("*.mat") if path.is_file())

    mat_by_key = {}
    for mat_path in mat_files:
        mat_by_key.setdefault(normalized_stem(mat_path), []).append(mat_path)

    matched_for_source = 0
    unmatched_for_source = 0

    for image_path in image_files:
        key = normalized_stem(image_path)
        selected_mat = None

        # 1) Exact same filename/stem in the same directory.
        direct_mat = image_path.with_suffix(".mat")
        if direct_mat.exists():
            selected_mat = direct_mat

        # 2) Normalized stem match in the same directory.
        if selected_mat is None:
            same_dir_matches = [
                p for p in mat_by_key.get(key, [])
                if p.parent == image_path.parent
            ]
            if len(same_dir_matches) == 1:
                selected_mat = same_dir_matches[0]

        # 3) One unique normalized-stem match anywhere in THIS source dataset.
        if selected_mat is None:
            candidates = mat_by_key.get(key, [])
            if len(candidates) == 1:
                selected_mat = candidates[0]

        # 4) Last fallback: one unique partial-name match inside THIS source only.
        if selected_mat is None:
            partial_matches = [
                p for p in mat_files
                if key in normalized_stem(p) or normalized_stem(p) in key
            ]
            if len(partial_matches) == 1:
                selected_mat = partial_matches[0]

        if selected_mat is not None:
            relative_image = image_path.relative_to(root)
            uid = f"{source}::{relative_image.as_posix()}"

            pairs.append({
                "source": source,
                "image": image_path,
                "mat": selected_mat,
                "key": key,
                "uid": uid,
                "relative_image": relative_image.as_posix(),
            })
            matched_for_source += 1
        else:
            unmatched_images.append({
                "source": source,
                "image": image_path,
            })
            unmatched_for_source += 1

    source_pair_stats.append({
        "source": source,
        "images": len(image_files),
        "mat_files": len(mat_files),
        "paired": matched_for_source,
        "unmatched": unmatched_for_source,
    })

print("PAIRING SUMMARY")
print("=" * 72)
for stat in source_pair_stats:
    print(
        f"{stat['source']}: images={stat['images']}, MAT={stat['mat_files']}, "
        f"paired={stat['paired']}, unmatched={stat['unmatched']}"
    )

print("\nTotal paired image/MAT records:", len(pairs))
print("Total unmatched images:", len(unmatched_images))

if unmatched_images:
    print("\nFirst unmatched images:")
    for item in unmatched_images[:20]:
        print(" -", item["source"], "|", item["image"])

if not pairs:
    raise RuntimeError(
        "No image/MAT pairs were found in either ZIP. Check the ZIP contents and annotation filenames."
    )

pair_summary = pd.DataFrame(source_pair_stats)
pair_summary.to_csv(RESULTS_ROOT / "pairing_summary.csv", index=False)

print("\nCell 3 completed successfully.")


In [ ]:
# CELL 4 — Robustly extract point coordinates from every MAT file

def collect_numeric_arrays(value, output):
    if value is None:
        return

    if isinstance(value, dict):
        for item in value.values():
            collect_numeric_arrays(item, output)
        return

    if hasattr(value, "_fieldnames"):
        for field_name in value._fieldnames:
            collect_numeric_arrays(
                getattr(value, field_name),
                output,
            )
        return

    if isinstance(value, np.ndarray):
        if value.dtype.names:
            for field_name in value.dtype.names:
                collect_numeric_arrays(
                    value[field_name],
                    output,
                )
            return

        if value.dtype == object:
            for item in value.flat:
                collect_numeric_arrays(item, output)
            return

        if np.issubdtype(value.dtype, np.number):
            output.append(np.asarray(value))
        return

    if isinstance(value, (list, tuple)):
        for item in value:
            collect_numeric_arrays(item, output)


def load_mat_flexible(mat_path):
    try:
        return loadmat(
            mat_path,
            simplify_cells=True,
            squeeze_me=True,
            struct_as_record=False,
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
        )


def candidate_point_arrays(mat_data):
    arrays = []
    collect_numeric_arrays(mat_data, arrays)

    candidates = []

    for array in arrays:
        array = np.asarray(array)

        if array.size < 2:
            continue

        array = np.squeeze(array)

        if array.ndim == 1:
            continue

        if array.ndim > 2:
            if array.shape[-1] >= 2:
                array = array.reshape(-1, array.shape[-1])
            elif array.shape[0] >= 2:
                array = np.moveaxis(array, 0, -1)
                array = array.reshape(-1, array.shape[-1])
            else:
                continue

        if array.ndim != 2:
            continue

        if array.shape[1] >= 2:
            points = array[:, :2]
        elif array.shape[0] >= 2:
            points = array[:2, :].T
        else:
            continue

        try:
            points = points.astype(np.float32)
        except Exception:
            continue

        points = points[
            np.isfinite(points).all(axis=1)
        ]

        if len(points) == 0:
            continue

        candidates.append(points)

    return candidates


def extract_points(mat_path, image_width, image_height):
    mat_data = load_mat_flexible(mat_path)
    candidates = candidate_point_arrays(mat_data)

    scored_candidates = []

    for points in candidates:
        for swap_xy in (False, True):
            candidate = (
                points[:, [1, 0]].copy()
                if swap_xy
                else points.copy()
            )

            x = candidate[:, 0]
            y = candidate[:, 1]

            valid = (
                (x >= 0)
                & (x < image_width)
                & (y >= 0)
                & (y < image_height)
            )

            valid_fraction = float(valid.mean())

            if valid_fraction < 0.70:
                continue

            filtered = candidate[valid]

            # Remove exact duplicate point entries.
            filtered = np.unique(
                np.round(filtered, 3),
                axis=0,
            ).astype(np.float32)

            if len(filtered) == 0:
                continue

            # Prefer arrays with many valid points and a high valid ratio.
            score = (
                valid_fraction,
                len(filtered),
            )

            scored_candidates.append(
                (score, filtered)
            )

    if not scored_candidates:
        return np.empty((0, 2), dtype=np.float32)

    scored_candidates.sort(
        key=lambda item: (
            item[0][0],
            item[0][1],
        ),
        reverse=True,
    )

    return scored_candidates[0][1]


records = []
failed_records = []

for index, pair in enumerate(pairs, start=1):
    try:
        with Image.open(pair["image"]) as image:
            width, height = image.size

        points = extract_points(
            pair["mat"],
            width,
            height,
        )

        records.append(
            {
                "uid": pair["uid"],
                "source": pair["source"],
                "image": str(pair["image"]),
                "mat": str(pair["mat"]),
                "width": width,
                "height": height,
                "points": points,
                "count": int(len(points)),
            }
        )

    except Exception as error:
        failed_records.append(
            {
                "image": str(pair["image"]),
                "mat": str(pair["mat"]),
                "error": str(error),
            }
        )

    if index % 50 == 0 or index == len(pairs):
        print(
            f"Parsed {index}/{len(pairs)} pairs"
        )


print("\nUsable records:", len(records))
print("Failed records:", len(failed_records))
print(
    "Total annotated tassels:",
    sum(record["count"] for record in records),
)

zero_count_records = [
    record
    for record in records
    if record["count"] == 0
]

print(
    "Records with zero extracted points:",
    len(zero_count_records),
)

if failed_records:
    print("\nFirst failed records:")
    for item in failed_records[:10]:
        print(item)

if not records:
    raise RuntimeError(
        "No usable image/MAT records were parsed."
    )


manifest = pd.DataFrame(
    [
        {
            "uid": record["uid"],
            "source": record["source"],
            "image": record["image"],
            "mat": record["mat"],
            "width": record["width"],
            "height": record["height"],
            "count": record["count"],
        }
        for record in records
    ]
)

manifest.to_csv(
    RESULTS_ROOT / "dataset_manifest.csv",
    index=False,
)

print("\nCount statistics (combined):")
print(manifest["count"].describe())

print("\nRecords by source:")
print(manifest.groupby("source").agg(images=("uid", "count"), tassels=("count", "sum"), mean_count=("count", "mean")))


In [ ]:
# CELL 5 — Visually verify extracted point annotations

import matplotlib.pyplot as plt

positive_records = [
    record
    for record in records
    if record["count"] > 0
]

if not positive_records:
    raise RuntimeError(
        "No positive point annotations were extracted."
    )

preview_records = random.Random(SEED).sample(
    positive_records,
    min(9, len(positive_records)),
)

figure, axes = plt.subplots(
    3,
    3,
    figsize=(16, 16),
)

axes = np.asarray(axes).reshape(-1)

for axis in axes:
    axis.axis("off")

for axis, record in zip(
    axes,
    preview_records,
):
    image = Image.open(
        record["image"]
    ).convert("RGB")

    points = record["points"]

    axis.imshow(image)
    axis.scatter(
        points[:, 0],
        points[:, 1],
        s=10,
        marker="x",
    )

    axis.set_title(
        f"{record['source']} | {Path(record['image']).name}\n"
        f"Count = {record['count']}",
        fontsize=9,
    )

    axis.axis("off")

plt.tight_layout()

preview_path = (
    RESULTS_ROOT
    / "point_annotation_preview.png"
)

plt.savefig(
    preview_path,
    dpi=170,
    bbox_inches="tight",
)

plt.show()

print("Preview saved:", preview_path)
print(
    "Check the plotted crosses before continuing. "
    "They must align with visible maize tassels."
)


In [ ]:
# CELL 6 — Create train, validation and independent test splits
# Stratify by source dataset so BOTH ZIP datasets are represented in each split.

indices = np.arange(len(records))
sources = np.array([record["source"] for record in records])


def usable_stratify(labels):
    values, counts = np.unique(labels, return_counts=True)
    return labels if len(values) > 1 and counts.min() >= 3 else None


first_stratify = usable_stratify(sources)

train_val_indices, test_indices = train_test_split(
    indices,
    test_size=0.15,
    random_state=SEED,
    shuffle=True,
    stratify=first_stratify,
)

train_val_sources = sources[train_val_indices]
second_stratify = usable_stratify(train_val_sources)

train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.1764705882,  # about 15% of the full combined dataset
    random_state=SEED,
    shuffle=True,
    stratify=second_stratify,
)

splits = {
    "train": [records[index] for index in train_indices],
    "val": [records[index] for index in val_indices],
    "test": [records[index] for index in test_indices],
}

split_rows = []

for split_name, split_records in splits.items():
    print(
        f"{split_name}: {len(split_records)} images, "
        f"{sum(record['count'] for record in split_records)} tassels"
    )

    source_counts = pd.Series(
        [record["source"] for record in split_records]
    ).value_counts()
    print(source_counts.to_string())
    print()

    for record in split_records:
        split_rows.append({
            "split": split_name,
            "uid": record["uid"],
            "source": record["source"],
            "image": record["image"],
            "count": record["count"],
        })

pd.DataFrame(split_rows).to_csv(
    RESULTS_ROOT / "dataset_splits.csv",
    index=False,
)

print("Cell 6 completed successfully.")


In [ ]:
# CELL 7 — Image, crop and density-map utilities

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32,
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32,
)


def resize_image_and_points(
    image,
    points,
    maximum_side,
):
    width, height = image.size

    scale = min(
        1.0,
        maximum_side / max(width, height),
    )

    if scale < 1.0:
        new_width = max(
            OUTPUT_STRIDE,
            int(round(width * scale)),
        )

        new_height = max(
            OUTPUT_STRIDE,
            int(round(height * scale)),
        )

        image = image.resize(
            (new_width, new_height),
            Image.Resampling.BILINEAR,
        )

        points = points.copy()
        points[:, 0] *= new_width / width
        points[:, 1] *= new_height / height

    return image, points


def make_density_map(
    points,
    output_height,
    output_width,
    stride=OUTPUT_STRIDE,
    sigma=GAUSSIAN_SIGMA,
):
    density = np.zeros(
        (output_height, output_width),
        dtype=np.float32,
    )

    if len(points) == 0:
        return density

    scaled_points = points / float(stride)

    for x, y in scaled_points:
        x_index = int(round(x))
        y_index = int(round(y))

        x_index = min(
            max(x_index, 0),
            output_width - 1,
        )

        y_index = min(
            max(y_index, 0),
            output_height - 1,
        )

        density[y_index, x_index] += 1.0

    original_count = float(
        density.sum()
    )

    density = gaussian_filter(
        density,
        sigma=sigma,
        mode="constant",
    )

    blurred_sum = float(
        density.sum()
    )

    if blurred_sum > 0:
        density *= (
            original_count / blurred_sum
        )

    return density.astype(np.float32)


def normalize_image(image_array):
    image_array = (
        image_array.astype(np.float32)
        / 255.0
    )

    image_array = (
        image_array - IMAGENET_MEAN
    ) / IMAGENET_STD

    image_array = np.transpose(
        image_array,
        (2, 0, 1),
    )

    return torch.from_numpy(
        image_array
    ).float()


def pad_image_and_points(
    image_array,
    points,
    minimum_height,
    minimum_width,
):
    height, width = image_array.shape[:2]

    padded_height = max(
        height,
        minimum_height,
    )

    padded_width = max(
        width,
        minimum_width,
    )

    canvas = np.full(
        (
            padded_height,
            padded_width,
            3,
        ),
        114,
        dtype=np.uint8,
    )

    canvas[:height, :width] = image_array

    return canvas, points


def pad_to_stride(
    image_array,
    stride=OUTPUT_STRIDE,
):
    height, width = image_array.shape[:2]

    padded_height = int(
        math.ceil(height / stride)
        * stride
    )

    padded_width = int(
        math.ceil(width / stride)
        * stride
    )

    canvas = np.full(
        (
            padded_height,
            padded_width,
            3,
        ),
        114,
        dtype=np.uint8,
    )

    canvas[:height, :width] = image_array

    return canvas

In [ ]:
# CELL 8 — Final stable training dataset and DataLoader

import random
import numpy as np
import torch

from PIL import Image, ImageEnhance
from torch.utils.data import Dataset, DataLoader


# ============================================================
# VERIFY OBJECTS CREATED IN PREVIOUS CELLS
# ============================================================

required_objects = [
    "splits",
    "resize_image_and_points",
    "pad_image_and_points",
    "make_density_map",
    "normalize_image",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Run Cells 1–7 before running Cell 8.\n"
        f"Missing objects: {missing_objects}"
    )


# ============================================================
# CONFIGURATION
# ============================================================

CROP_SIZE = globals().get(
    "CROP_SIZE",
    512,
)

OUTPUT_STRIDE = globals().get(
    "OUTPUT_STRIDE",
    8,
)

CROPS_PER_IMAGE = globals().get(
    "CROPS_PER_IMAGE",
    3,
)

TRAIN_MAX_SIDE = globals().get(
    "TRAIN_MAX_SIDE",
    1600,
)

POINT_CENTERED_CROP_PROBABILITY = globals().get(
    "POINT_CENTERED_CROP_PROBABILITY",
    0.70,
)

HORIZONTAL_FLIP_PROBABILITY = globals().get(
    "HORIZONTAL_FLIP_PROBABILITY",
    0.50,
)

BATCH_SIZE = 4

# Use zero workers to prevent Kaggle DataLoader crashes.
NUM_WORKERS = 0

PIN_MEMORY = torch.cuda.is_available()

print("Crop size          :", CROP_SIZE)
print("Output stride      :", OUTPUT_STRIDE)
print("Crops per image    :", CROPS_PER_IMAGE)
print("Batch size         :", BATCH_SIZE)
print("DataLoader workers :", NUM_WORKERS)
print("Pin memory         :", PIN_MEMORY)


# ============================================================
# DATASET
# ============================================================

class MaizeDensityDataset(Dataset):

    def __init__(
        self,
        records,
        crops_per_image=3,
        augment=True,
    ):
        self.records = records
        self.crops_per_image = crops_per_image
        self.augment = augment

        if len(self.records) == 0:
            raise ValueError(
                "The dataset received no records."
            )

    def __len__(self):
        return (
            len(self.records)
            * self.crops_per_image
        )

    def __getitem__(self, index):

        # Repeat every full image multiple times with
        # independently generated random crops.
        record_index = (
            index % len(self.records)
        )

        record = self.records[
            record_index
        ]

        # ----------------------------------------------------
        # Read image and annotations
        # ----------------------------------------------------

        image = Image.open(
            record["image"]
        ).convert("RGB")

        points = np.asarray(
            record["points"],
            dtype=np.float32,
        ).copy()

        # Guarantee shape (N, 2), including empty annotations.
        if points.size == 0:
            points = np.empty(
                (0, 2),
                dtype=np.float32,
            )
        else:
            points = points.reshape(
                -1,
                2,
            )

        # ----------------------------------------------------
        # Resize large images
        # ----------------------------------------------------

        image, points = resize_image_and_points(
            image,
            points,
            TRAIN_MAX_SIDE,
        )

        # ----------------------------------------------------
        # Photometric augmentation
        # ----------------------------------------------------

        if self.augment:

            brightness_factor = random.uniform(
                0.85,
                1.15,
            )

            contrast_factor = random.uniform(
                0.85,
                1.15,
            )

            image = ImageEnhance.Brightness(
                image
            ).enhance(
                brightness_factor
            )

            image = ImageEnhance.Contrast(
                image
            ).enhance(
                contrast_factor
            )

        image_array = np.asarray(
            image,
            dtype=np.uint8,
        )

        # ----------------------------------------------------
        # Pad images smaller than the crop
        # ----------------------------------------------------

        image_array, points = pad_image_and_points(
            image_array,
            points,
            minimum_height=CROP_SIZE,
            minimum_width=CROP_SIZE,
        )

        image_height, image_width = (
            image_array.shape[:2]
        )

        maximum_x0 = max(
            image_width - CROP_SIZE,
            0,
        )

        maximum_y0 = max(
            image_height - CROP_SIZE,
            0,
        )

        # ----------------------------------------------------
        # Select crop position
        # ----------------------------------------------------

        use_point_centered_crop = (
            len(points) > 0
            and random.random()
            < POINT_CENTERED_CROP_PROBABILITY
        )

        if use_point_centered_crop:

            selected_index = random.randrange(
                len(points)
            )

            selected_x, selected_y = points[
                selected_index
            ]

            horizontal_position = random.uniform(
                0.25,
                0.75,
            )

            vertical_position = random.uniform(
                0.25,
                0.75,
            )

            x0 = int(
                round(
                    selected_x
                    - horizontal_position
                    * CROP_SIZE
                )
            )

            y0 = int(
                round(
                    selected_y
                    - vertical_position
                    * CROP_SIZE
                )
            )

            x0 = min(
                max(x0, 0),
                maximum_x0,
            )

            y0 = min(
                max(y0, 0),
                maximum_y0,
            )

        else:

            x0 = (
                random.randint(
                    0,
                    maximum_x0,
                )
                if maximum_x0 > 0
                else 0
            )

            y0 = (
                random.randint(
                    0,
                    maximum_y0,
                )
                if maximum_y0 > 0
                else 0
            )

        # ----------------------------------------------------
        # Extract image crop
        # ----------------------------------------------------

        crop = image_array[
            y0:y0 + CROP_SIZE,
            x0:x0 + CROP_SIZE,
        ]

        crop = np.ascontiguousarray(
            crop
        )

        # ----------------------------------------------------
        # Transform point coordinates into crop coordinates
        # ----------------------------------------------------

        crop_points = points.copy()

        if len(crop_points) > 0:

            crop_points[:, 0] -= x0
            crop_points[:, 1] -= y0

            points_inside_crop = (
                (crop_points[:, 0] >= 0)
                & (
                    crop_points[:, 0]
                    < CROP_SIZE
                )
                & (crop_points[:, 1] >= 0)
                & (
                    crop_points[:, 1]
                    < CROP_SIZE
                )
            )

            crop_points = crop_points[
                points_inside_crop
            ]

        # ----------------------------------------------------
        # Horizontal flipping
        # ----------------------------------------------------

        if (
            self.augment
            and random.random()
            < HORIZONTAL_FLIP_PROBABILITY
        ):

            crop = np.ascontiguousarray(
                crop[:, ::-1, :]
            )

            if len(crop_points) > 0:
                crop_points[:, 0] = (
                    CROP_SIZE
                    - 1
                    - crop_points[:, 0]
                )

        # ----------------------------------------------------
        # Generate density map
        # ----------------------------------------------------

        output_height = (
            CROP_SIZE // OUTPUT_STRIDE
        )

        output_width = (
            CROP_SIZE // OUTPUT_STRIDE
        )

        density_map = make_density_map(
            crop_points,
            output_height=output_height,
            output_width=output_width,
            stride=OUTPUT_STRIDE,
        )

        # ----------------------------------------------------
        # Convert to PyTorch tensors
        # ----------------------------------------------------

        image_tensor = normalize_image(
            crop
        )

        density_tensor = torch.from_numpy(
            np.asarray(
                density_map,
                dtype=np.float32,
            )
        ).unsqueeze(0)

        count_tensor = torch.tensor(
            float(len(crop_points)),
            dtype=torch.float32,
        )

        return (
            image_tensor,
            density_tensor,
            count_tensor,
        )


# ============================================================
# CREATE TRAINING DATASET
# ============================================================

train_dataset = MaizeDensityDataset(
    records=splits["train"],
    crops_per_image=CROPS_PER_IMAGE,
    augment=True,
)


# ============================================================
# CREATE STABLE KAGGLE DATALOADER
# ============================================================

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=PIN_MEMORY,
    drop_last=True,
    persistent_workers=False,
)


# ============================================================
# BASIC INFORMATION
# ============================================================

print("\nTraining images  :", len(splits["train"]))
print("Dataset samples  :", len(train_dataset))
print("Batches per epoch:", len(train_loader))


# ============================================================
# TEST DATALOADER BEFORE TRAINING
# ============================================================

try:

    test_batch = next(
        iter(train_loader)
    )

    (
        test_images,
        test_density_maps,
        test_counts,
    ) = test_batch

    print("\nDataLoader test successful.")
    print(
        "Image tensor shape  :",
        tuple(test_images.shape),
    )

    print(
        "Density tensor shape:",
        tuple(test_density_maps.shape),
    )

    print(
        "Count tensor shape  :",
        tuple(test_counts.shape),
    )

    print(
        "Batch counts        :",
        test_counts.tolist(),
    )

    density_sums = (
        test_density_maps
        .flatten(start_dim=1)
        .sum(dim=1)
    )

    print(
        "Density-map sums    :",
        density_sums.tolist(),
    )

    maximum_difference = torch.max(
        torch.abs(
            density_sums
            - test_counts
        )
    ).item()

    print(
        "Maximum count difference:",
        maximum_difference,
    )

    if maximum_difference > 0.10:
        print(
            "\nWarning: density-map sums do not closely "
            "match the point counts."
        )
    else:
        print(
            "\nDensity-map count verification passed."
        )

except Exception as error:

    raise RuntimeError(
        "The DataLoader test failed. "
        "Check the image and MAT annotation records."
    ) from error


print("\nCell 8 completed successfully.")

In [ ]:
# CELL 9 — Final CSRNet-style density-counting network

import torch
import torch.nn as nn

# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# Build backend layers
# ------------------------------------------------------------
def make_backend(
    input_channels,
    output_channels_list,
    dilation=True,
):
    layers = []

    for output_channels in output_channels_list:
        layers.append(
            nn.Conv2d(
                in_channels=input_channels,
                out_channels=output_channels,
                kernel_size=3,
                padding=2 if dilation else 1,
                dilation=2 if dilation else 1,
            )
        )

        layers.append(
            nn.ReLU(inplace=True)
        )

        input_channels = output_channels

    return nn.Sequential(*layers)


# ------------------------------------------------------------
# Manual VGG-style frontend fallback
# Output stride = 8
# ------------------------------------------------------------
def make_manual_vgg_frontend():
    layers = []

    configuration = [
        64, 64, "M",
        128, 128, "M",
        256, 256, 256, "M",
        512, 512, 512,
    ]

    input_channels = 3

    for item in configuration:
        if item == "M":
            layers.append(
                nn.MaxPool2d(
                    kernel_size=2,
                    stride=2,
                )
            )
        else:
            layers.append(
                nn.Conv2d(
                    input_channels,
                    item,
                    kernel_size=3,
                    padding=1,
                )
            )

            layers.append(
                nn.ReLU(inplace=True)
            )

            input_channels = item

    return nn.Sequential(*layers)


# ------------------------------------------------------------
# CSRNet model
# ------------------------------------------------------------
class CSRNetLite(nn.Module):

    def __init__(self, pretrained=True):
        super().__init__()

        self.pretrained_frontend_loaded = False

        # Try loading pretrained VGG16
        if pretrained:
            try:
                from torchvision.models import (
                    vgg16,
                    VGG16_Weights,
                )

                vgg_model = vgg16(
                    weights=VGG16_Weights.IMAGENET1K_V1
                )

                # VGG16 through relu4_3.
                # Output stride is 8 and channels are 512.
                self.frontend = nn.Sequential(
                    *list(vgg_model.features.children())[:23]
                )

                self.pretrained_frontend_loaded = True

                print(
                    "Loaded ImageNet-pretrained VGG16 frontend."
                )

            except Exception as error:
                print(
                    "Could not load torchvision pretrained VGG16:"
                )
                print(error)

                print(
                    "Using manually created VGG-style frontend."
                )

                self.frontend = make_manual_vgg_frontend()

        else:
            self.frontend = make_manual_vgg_frontend()

        # Dilated backend
        self.backend = make_backend(
            input_channels=512,
            output_channels_list=[
                512,
                512,
                256,
                128,
                64,
            ],
            dilation=True,
        )

        # One-channel density output
        self.output_layer = nn.Sequential(
            nn.Conv2d(
                in_channels=64,
                out_channels=1,
                kernel_size=1,
            ),

            # Ensures non-negative density values
            nn.Softplus(
                beta=1.0,
                threshold=20.0,
            ),
        )

        self.initialize_new_layers()

    def initialize_new_layers(self):

        # Initialize the manual frontend only when pretrained
        # weights were not loaded.
        if not self.pretrained_frontend_loaded:
            for module in self.frontend.modules():
                if isinstance(module, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        module.weight,
                        mode="fan_out",
                        nonlinearity="relu",
                    )

                    if module.bias is not None:
                        nn.init.constant_(
                            module.bias,
                            0.0,
                        )

        # Initialize CSRNet backend
        for module in self.backend.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.01,
                )

                if module.bias is not None:
                    nn.init.constant_(
                        module.bias,
                        0.0,
                    )

        # Initialize density output layer
        for module in self.output_layer.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.01,
                )

                if module.bias is not None:
                    nn.init.constant_(
                        module.bias,
                        0.0,
                    )

    def forward(self, images):

        features = self.frontend(images)
        features = self.backend(features)
        density_map = self.output_layer(features)

        return density_map


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------
model = CSRNetLite(
    pretrained=True
).to(DEVICE)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Trainable parameters:",
    f"{trainable_parameters / 1e6:.2f} million"
)

print(
    "Total parameters:",
    f"{total_parameters / 1e6:.2f} million"
)


# ------------------------------------------------------------
# Verify input/output dimensions
# ------------------------------------------------------------
model.eval()

with torch.no_grad():

    test_input = torch.zeros(
        1,
        3,
        512,
        512,
        device=DEVICE,
    )

    test_output = model(test_input)

print("Test input shape :", tuple(test_input.shape))
print("Density-map shape:", tuple(test_output.shape))

expected_shape = (
    1,
    1,
    512 // OUTPUT_STRIDE,
    512 // OUTPUT_STRIDE,
)

if tuple(test_output.shape) != expected_shape:
    raise RuntimeError(
        "Model output shape is incorrect.\n"
        f"Expected: {expected_shape}\n"
        f"Received: {tuple(test_output.shape)}"
    )

print("Cell 9 completed successfully.")

In [ ]:
# CELL 10 — Full-image prediction and evaluation functions

@torch.no_grad()
def predict_record_count(
    model,
    record,
    horizontal_flip_tta=True,
):
    image = Image.open(
        record["image"]
    ).convert("RGB")

    points = record[
        "points"
    ].copy().astype(np.float32)

    image, _ = resize_image_and_points(
        image,
        points,
        EVAL_MAX_SIDE,
    )

    image_array = np.asarray(
        image,
        dtype=np.uint8,
    )

    image_array = pad_to_stride(
        image_array,
        OUTPUT_STRIDE,
    )

    image_tensor = normalize_image(
        image_array
    ).unsqueeze(0).to(
        DEVICE,
        non_blocking=True,
    )

    model.eval()

    density = model(
        image_tensor
    )

    predicted_count = float(
        density.sum().item()
    )

    if horizontal_flip_tta:
        flipped_tensor = torch.flip(
            image_tensor,
            dims=[3],
        )

        flipped_density = model(
            flipped_tensor
        )

        flipped_count = float(
            flipped_density.sum().item()
        )

        predicted_count = (
            predicted_count
            + flipped_count
        ) / 2.0

    return predicted_count


def calculate_count_metrics(
    actual_counts,
    predicted_counts,
):
    actual = np.asarray(
        actual_counts,
        dtype=np.float64,
    )

    predicted = np.asarray(
        predicted_counts,
        dtype=np.float64,
    )

    errors = predicted - actual

    mae = float(
        np.mean(np.abs(errors))
    )

    rmse = float(
        np.sqrt(np.mean(errors ** 2))
    )

    weighted_accuracy = 100.0 * max(
        0.0,
        1.0
        - float(
            np.sum(np.abs(errors))
        )
        / max(
            float(np.sum(actual)),
            1.0,
        ),
    )

    total_variation = float(
        np.sum(
            (
                actual
                - actual.mean()
            )
            ** 2
        )
    )

    r_squared = (
        float(
            1.0
            - np.sum(errors ** 2)
            / total_variation
        )
        if total_variation > 1e-12
        else float("nan")
    )

    tolerance = np.maximum(
        1.0,
        0.10 * actual,
    )

    within_ten_percent = 100.0 * float(
        np.mean(
            np.abs(errors)
            <= tolerance
        )
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r_squared,
        "weighted_count_accuracy": weighted_accuracy,
        "images_within_10_percent": within_ten_percent,
    }


@torch.no_grad()
def evaluate_records(
    model,
    records_to_evaluate,
    split_name,
    verbose=False,
):
    rows = []

    for index, record in enumerate(
        records_to_evaluate,
        start=1,
    ):
        prediction = predict_record_count(
            model,
            record,
            horizontal_flip_tta=True,
        )

        rows.append(
            {
                "uid": record["uid"],
                "image": record["image"],
                "ground_truth": record["count"],
                "prediction": prediction,
                "error": prediction - record["count"],
            }
        )

        if verbose:
            print(
                f"{split_name} "
                f"{index}/{len(records_to_evaluate)} | "
                f"GT={record['count']} | "
                f"Pred={prediction:.2f}"
            )

    dataframe = pd.DataFrame(rows)

    metrics = calculate_count_metrics(
        dataframe[
            "ground_truth"
        ].to_numpy(),
        dataframe[
            "prediction"
        ].to_numpy(),
    )

    return dataframe, metrics

In [ ]:
# CELL 11 — Final training loop with iterations and early stopping

from pathlib import Path
import pandas as pd
import torch
import torch.nn.functional as F

# ============================================================
# VERIFY REQUIRED OBJECTS FROM PREVIOUS CELLS
# ============================================================

required_objects = [
    "model",
    "train_loader",
    "evaluate_records",
    "splits",
    "DEVICE",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Run Cells 1–10 before Cell 11.\n"
        f"Missing objects: {missing_objects}"
    )

# ============================================================
# TRAINING CONFIGURATION
# ============================================================

RESULTS_ROOT = Path(
    globals().get(
        "RESULTS_ROOT",
        "/kaggle/working/maize_tassel_counting_results",
    )
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Maximum epochs
EPOCHS = globals().get("EPOCHS", 120)

# Stop when validation MAE does not improve for these epochs
EARLY_STOPPING = globals().get(
    "EARLY_STOPPING",
    15,
)

# None means use every batch in each epoch.
# Use 10 or 20 only for quick code testing.
MAX_ITERATIONS_PER_EPOCH = None

# Print training progress after every N iterations
PRINT_EVERY = 20

DENSITY_SCALE = globals().get(
    "DENSITY_SCALE",
    100.0,
)

FRONTEND_LR = globals().get(
    "FRONTEND_LR",
    1e-5,
)

BACKEND_LR = globals().get(
    "BACKEND_LR",
    1e-4,
)

WEIGHT_DECAY = globals().get(
    "WEIGHT_DECAY",
    1e-4,
)

COUNT_LOSS_WEIGHT = 0.01
MINIMUM_IMPROVEMENT = 1e-4

BEST_MODEL_PATH = (
    RESULTS_ROOT
    / "best_density_counter.pt"
)

LAST_MODEL_PATH = (
    RESULTS_ROOT
    / "last_density_counter.pt"
)

HISTORY_PATH = (
    RESULTS_ROOT
    / "training_history.csv"
)

print("Maximum epochs             :", EPOCHS)
print("Early-stopping patience    :", EARLY_STOPPING)
print("Maximum iterations/epoch   :", MAX_ITERATIONS_PER_EPOCH)
print("Best model path            :", BEST_MODEL_PATH)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.frontend.parameters(),
            "lr": FRONTEND_LR,
        },
        {
            "params": model.backend.parameters(),
            "lr": BACKEND_LR,
        },
        {
            "params": model.output_layer.parameters(),
            "lr": BACKEND_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

# Reduce learning rate when validation MAE stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-7,
)

# ============================================================
# MIXED-PRECISION TRAINING
# ============================================================

USE_AMP = DEVICE.type == "cuda"

try:
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )
except Exception:
    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP,
    )

print("Mixed precision enabled:", USE_AMP)

# ============================================================
# TRAINING STATE
# ============================================================

best_validation_mae = float("inf")
epochs_without_improvement = 0
global_iteration = 0
history_rows = []

# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(1, EPOCHS + 1):

    model.train()

    running_total_loss = 0.0
    running_density_loss = 0.0
    running_count_loss = 0.0

    completed_iterations = 0

    for iteration, batch in enumerate(
        train_loader,
        start=1,
    ):

        # Optionally limit batches in each epoch
        if (
            MAX_ITERATIONS_PER_EPOCH is not None
            and iteration > MAX_ITERATIONS_PER_EPOCH
        ):
            break

        (
            images_batch,
            density_batch,
            count_batch,
        ) = batch

        images_batch = images_batch.to(
            DEVICE,
            non_blocking=True,
        )

        density_batch = density_batch.to(
            DEVICE,
            non_blocking=True,
        )

        count_batch = count_batch.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True,
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):

            predicted_density = model(
                images_batch
            )

            # Verify model and target dimensions
            if (
                predicted_density.shape
                != density_batch.shape
            ):
                density_batch = F.interpolate(
                    density_batch,
                    size=predicted_density.shape[-2:],
                    mode="bilinear",
                    align_corners=False,
                )

            density_loss = F.mse_loss(
                predicted_density
                * DENSITY_SCALE,
                density_batch
                * DENSITY_SCALE,
            )

            predicted_counts = (
                predicted_density
                .flatten(start_dim=1)
                .sum(dim=1)
            )

            count_loss = F.smooth_l1_loss(
                predicted_counts,
                count_batch,
            )

            total_loss = (
                density_loss
                + COUNT_LOSS_WEIGHT
                * count_loss
            )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        scaler.scale(
            total_loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        # ----------------------------------------------------
        # Update iteration statistics
        # ----------------------------------------------------

        global_iteration += 1
        completed_iterations += 1

        running_total_loss += float(
            total_loss.item()
        )

        running_density_loss += float(
            density_loss.item()
        )

        running_count_loss += float(
            count_loss.item()
        )

        if (
            iteration % PRINT_EVERY == 0
            or iteration == 1
        ):
            batch_actual_count = float(
                count_batch.mean().item()
            )

            batch_predicted_count = float(
                predicted_counts.mean().item()
            )

            print(
                f"Epoch {epoch:03d}/{EPOCHS} | "
                f"Iteration {iteration:04d}/"
                f"{len(train_loader):04d} | "
                f"Global iteration {global_iteration:06d} | "
                f"Loss={total_loss.item():.4f} | "
                f"Actual count={batch_actual_count:.2f} | "
                f"Predicted count={batch_predicted_count:.2f}"
            )

    # Avoid division by zero
    completed_iterations = max(
        completed_iterations,
        1,
    )

    average_total_loss = (
        running_total_loss
        / completed_iterations
    )

    average_density_loss = (
        running_density_loss
        / completed_iterations
    )

    average_count_loss = (
        running_count_loss
        / completed_iterations
    )

    # ========================================================
    # VALIDATION AFTER EACH EPOCH
    # ========================================================

    model.eval()

    (
        validation_predictions,
        validation_metrics,
    ) = evaluate_records(
        model,
        splits["val"],
        split_name="validation",
        verbose=False,
    )

    validation_mae = float(
        validation_metrics["MAE"]
    )

    validation_rmse = float(
        validation_metrics["RMSE"]
    )

    validation_accuracy = float(
        validation_metrics[
            "weighted_count_accuracy"
        ]
    )

    scheduler.step(
        validation_mae
    )

    frontend_lr = optimizer.param_groups[
        0
    ]["lr"]

    backend_lr = optimizer.param_groups[
        1
    ]["lr"]

    # ========================================================
    # CHECK FOR IMPROVEMENT
    # ========================================================

    improved = (
        validation_mae
        < best_validation_mae
        - MINIMUM_IMPROVEMENT
    )

    checkpoint = {
        "epoch": epoch,
        "global_iteration": global_iteration,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "validation_metrics": validation_metrics,
        "best_validation_mae": min(
            best_validation_mae,
            validation_mae,
        ),
        "configuration": {
            "epochs": EPOCHS,
            "early_stopping": EARLY_STOPPING,
            "density_scale": DENSITY_SCALE,
            "count_loss_weight": COUNT_LOSS_WEIGHT,
            "frontend_lr": FRONTEND_LR,
            "backend_lr": BACKEND_LR,
        },
    }

    # Always save the latest checkpoint
    torch.save(
        checkpoint,
        LAST_MODEL_PATH,
    )

    if improved:

        best_validation_mae = validation_mae
        epochs_without_improvement = 0

        torch.save(
            checkpoint,
            BEST_MODEL_PATH,
        )

        best_status = "BEST"

    else:

        epochs_without_improvement += 1
        best_status = ""

    # ========================================================
    # SAVE TRAINING HISTORY
    # ========================================================

    history_rows.append(
        {
            "epoch": epoch,
            "global_iteration": global_iteration,
            "completed_iterations": completed_iterations,
            "train_loss": average_total_loss,
            "density_loss": average_density_loss,
            "count_loss": average_count_loss,
            "validation_MAE": validation_mae,
            "validation_RMSE": validation_rmse,
            "validation_accuracy": validation_accuracy,
            "best_validation_MAE": best_validation_mae,
            "epochs_without_improvement": epochs_without_improvement,
            "frontend_lr": frontend_lr,
            "backend_lr": backend_lr,
        }
    )

    pd.DataFrame(
        history_rows
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    # Save validation predictions from current epoch
    validation_predictions.to_csv(
        RESULTS_ROOT
        / "latest_validation_predictions.csv",
        index=False,
    )

    # ========================================================
    # PRINT EPOCH RESULT
    # ========================================================

    print(
        "\n"
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Iterations={completed_iterations} | "
        f"Loss={average_total_loss:.4f} | "
        f"Val MAE={validation_mae:.3f} | "
        f"Val RMSE={validation_rmse:.3f} | "
        f"Val Acc={validation_accuracy:.2f}% | "
        f"No improvement="
        f"{epochs_without_improvement}/"
        f"{EARLY_STOPPING} | "
        f"{best_status}"
    )

    print("-" * 110)

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >= EARLY_STOPPING
    ):

        print(
            f"\nEarly stopping activated at epoch {epoch}."
        )

        print(
            f"Best validation MAE: "
            f"{best_validation_mae:.3f}"
        )

        print(
            f"Best checkpoint: {BEST_MODEL_PATH}"
        )

        break

# ============================================================
# TRAINING COMPLETE
# ============================================================

print("\nTraining completed.")
print("Best validation MAE:", best_validation_mae)
print("Best model:", BEST_MODEL_PATH)
print("Last model:", LAST_MODEL_PATH)
print("Training history:", HISTORY_PATH)

In [ ]:
# CELL 12 — Final independent test evaluation

checkpoint = torch.load(
    RESULTS_ROOT / "best_density_counter.pt",
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

validation_predictions, validation_metrics = evaluate_records(
    model,
    splits["val"],
    split_name="validation",
    verbose=True,
)

test_predictions, test_metrics = evaluate_records(
    model,
    splits["test"],
    split_name="test",
    verbose=True,
)

validation_predictions.to_csv(
    RESULTS_ROOT / "validation_predictions.csv",
    index=False,
)

test_predictions.to_csv(
    RESULTS_ROOT / "test_predictions.csv",
    index=False,
)

with open(
    RESULTS_ROOT / "validation_metrics.json",
    "w",
) as file:
    json.dump(
        validation_metrics,
        file,
        indent=2,
    )

with open(
    RESULTS_ROOT / "test_metrics.json",
    "w",
) as file:
    json.dump(
        test_metrics,
        file,
        indent=2,
    )

print()
print("FINAL TEST METRICS")

for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.6f}")

if test_metrics["weighted_count_accuracy"] >= 90.0:
    print()
    print("TARGET ACHIEVED: test counting accuracy is at least 90%.")
else:
    print()
    print(
        "The independent test set is below 90%. "
        "Review point extraction and annotation preview before tuning."
    )


In [ ]:
# CELL 13 — Save training and counting plots

history = pd.read_csv(
    RESULTS_ROOT / "training_history.csv"
)

# Cell 11 writes validation_MAE. Older notebook versions wrote val_MAE.
mae_column = "validation_MAE" if "validation_MAE" in history.columns else "val_MAE"
accuracy_column = (
    "validation_accuracy" if "validation_accuracy" in history.columns
    else "val_weighted_accuracy"
)

plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history["train_loss"])
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training loss")
plt.tight_layout()
plt.savefig(
    RESULTS_ROOT / "training_loss.png",
    dpi=170,
    bbox_inches="tight",
)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history[mae_column])
plt.xlabel("Epoch")
plt.ylabel("Validation MAE")
plt.title("Validation counting MAE")
plt.tight_layout()
plt.savefig(
    RESULTS_ROOT / "validation_mae.png",
    dpi=170,
    bbox_inches="tight",
)
plt.show()

if accuracy_column in history.columns:
    plt.figure(figsize=(8, 5))
    plt.plot(history["epoch"], history[accuracy_column])
    plt.xlabel("Epoch")
    plt.ylabel("Validation weighted count accuracy (%)")
    plt.title("Validation counting accuracy")
    plt.tight_layout()
    plt.savefig(
        RESULTS_ROOT / "validation_accuracy.png",
        dpi=170,
        bbox_inches="tight",
    )
    plt.show()

actual = test_predictions["ground_truth"].to_numpy(dtype=float)
predicted = test_predictions["prediction"].to_numpy(dtype=float)

maximum_count = max(
    float(actual.max()),
    float(predicted.max()),
    1.0,
)

plt.figure(figsize=(7, 6))
plt.scatter(actual, predicted, alpha=0.70)
plt.plot([0, maximum_count], [0, maximum_count], "--")
plt.xlabel("Ground-truth tassel count")
plt.ylabel("Predicted tassel count")
plt.title(
    "Independent test counting accuracy: "
    f"{test_metrics['weighted_count_accuracy']:.2f}%"
)
plt.tight_layout()
plt.savefig(
    RESULTS_ROOT / "test_count_scatter.png",
    dpi=170,
    bbox_inches="tight",
)
plt.show()

# Add dataset/source names to test predictions for per-dataset reporting.
source_by_uid = {record["uid"]: record["source"] for record in splits["test"]}
test_predictions["source"] = test_predictions["uid"].map(source_by_uid)
test_predictions.to_csv(RESULTS_ROOT / "test_predictions.csv", index=False)

source_metrics_rows = []
for source_name, group in test_predictions.groupby("source"):
    metrics = calculate_count_metrics(
        group["ground_truth"].to_numpy(),
        group["prediction"].to_numpy(),
    )
    source_metrics_rows.append({"source": source_name, **metrics, "images": len(group)})

source_metrics_df = pd.DataFrame(source_metrics_rows)
source_metrics_df.to_csv(RESULTS_ROOT / "test_metrics_by_dataset.csv", index=False)

print("\nTEST METRICS BY DATASET")
print(source_metrics_df.to_string(index=False))


In [ ]:
# CELL 14 — Save final summary and downloadable ZIP

dataset_summary = [
    {
        "source": dataset["source"],
        "zip_path": str(dataset["zip_path"]),
        "extracted_root": str(dataset["root"]),
        "image_count": int(dataset["image_count"]),
        "mat_count": int(dataset["mat_count"]),
    }
    for dataset in DATASETS
]

final_summary = {
    "datasets": dataset_summary,
    "number_of_combined_records": len(records),
    "train_images": len(splits["train"]),
    "validation_images": len(splits["val"]),
    "test_images": len(splits["test"]),
    "model": "CSRNetLite with VGG16 frontend",
    "crop_size": CROP_SIZE,
    "evaluation_max_side": EVAL_MAX_SIDE,
    "best_validation_metrics": validation_metrics,
    "independent_test_metrics": test_metrics,
    "best_model": str(RESULTS_ROOT / "best_density_counter.pt"),
}

with open(
    RESULTS_ROOT / "final_summary.json",
    "w",
) as file:
    json.dump(final_summary, file, indent=2)

archive_path = shutil.make_archive(
    "/kaggle/working/maize_tassel_counting_results",
    "zip",
    RESULTS_ROOT,
)

print("\nFINAL FILES")
print("Best model:", RESULTS_ROOT / "best_density_counter.pt")
print("Test metrics:", RESULTS_ROOT / "test_metrics.json")
print("Test metrics by dataset:", RESULTS_ROOT / "test_metrics_by_dataset.csv")
print("Test predictions:", RESULTS_ROOT / "test_predictions.csv")
print("Downloadable ZIP:", archive_path)
print(
    "\nUse Save Version in Kaggle, then download "
    "maize_tassel_counting_results.zip from the notebook Output section."
)


In [ ]:
# CELL 15 — OPTIONAL: predict one new image

NEW_IMAGE_PATH = Path(
    "/kaggle/working/new_maize_image.jpg"
)

if not NEW_IMAGE_PATH.exists():
    print(
        "Optional prediction skipped. "
        "Upload an image to:",
        NEW_IMAGE_PATH,
    )
else:
    temporary_record = {
        "image": str(NEW_IMAGE_PATH),
        "points": np.empty(
            (0, 2),
            dtype=np.float32,
        ),
    }

    predicted_count = predict_record_count(
        model,
        temporary_record,
        horizontal_flip_tta=True,
    )

    print(
        "Predicted maize tassel count:",
        round(predicted_count),
    )
    print(
        "Raw model count:",
        f"{predicted_count:.3f}",
    )